*Notebook Last Validated: 2026-11-04*

Project UID (Internal, Prod): a8d4dfc8-fcca-414d-a631-a4ae636263f6

# Tutorial 4 — OMOP Harmonization with the Rhino DHE

This notebook maps the three prepared datasets to OMOP CDM target tables:
- `patients` (prepared) → **OMOP Person**
- `encounters` (prepared) → **OMOP Visit Occurrence**
- `procedures` (prepared) → **OMOP Procedure Occurrence**

**Can I do this in the UI instead?**
Yes — all harmonization steps (creating mappings, approving semantic terms,
running harmonization) can be done via the FCP Dashboard at
https://dashboard.rhinohealth.com/login under Projects → Data Mappings.
The notebook automates those same steps.

**What standard are we using?**
This tutorial uses **OMOP CDM**. The Rhino DHE also supports **FHIR** and
**custom target models** — only the `target_data_model_type` parameter and
the target schema UIDs change. The full workflow structure is identical.

---
**Prerequisites:**
- Tutorial 3 complete — all six prepared UIDs required below
- OMOP target schemas pre-defined in the FCP (person, visit_occurrence, procedure_occurrence)
- Vocabulary UIDs for Gender, Race, Ethnicity, Visit Type, CPT4 (retrieved in Step 2)

**Inputs:** Prepared datasets and schemas from Tutorial 3

**Outputs:** Three OMOP-conformant datasets on the Rhino client, plus the
reusable semantic and syntactic mapping objects stored in the FCP

## Configuration

In [1]:
import os
import time
import rhino_health as rh
from getpass import getpass
from rhino_health import ApiEnvironment

# From Tutorial 3 — paste your values here
# PROJECT_UID = "<YOUR_PROJECT_UID>"  # REPLACE with your project UID"

# PREPARED_PATIENTS_UID          = "<PREPARED_PATIENTS_UID>"          # REPLACE with your Patients dataset UID
# PREPARED_ENCOUNTERS_UID        = "<PREPARED_ENCOUNTERS_UID>"        # REPLACE with your Encounters dataset UID
# PREPARED_PROCEDURES_UID        = "<PREPARED_PROCEDURES_UID>"        # REPLACE with your Procedures dataset UID
# PREPARED_PATIENTS_SCHEMA_UID   = "<PREPARED_PATIENTS_SCHEMA_UID>"   # REPLACE with your Patients schema UID
# PREPARED_ENCOUNTERS_SCHEMA_UID = "<PREPARED_ENCOUNTERS_SCHEMA_UID>" # REPLACE with your Encounters schema UID
# PREPARED_PROCEDURES_SCHEMA_UID = "<PREPARED_PROCEDURES_SCHEMA_UID>" # REPLACE with your Procedures schema UID

PROJECT_UID = "a8d4dfc8-fcca-414d-a631-a4ae636263f6"

PREPARED_PATIENTS_UID          = 'aee36141-c6a4-4e8a-8329-842f99d3c491'
PREPARED_ENCOUNTERS_UID        = 'f33e90c7-95a9-454a-8c17-fd9a60ead9b0'
PREPARED_PROCEDURES_UID        = '75b9e261-354f-4c0b-89ec-c0fd3aa9c85b'
PREPARED_PATIENTS_SCHEMA_UID   = '267bffca-3f53-4012-8768-242b8647cbf4'
PREPARED_ENCOUNTERS_SCHEMA_UID = '82573bef-1b8a-4b12-bca6-9ac6989b7920'
PREPARED_PROCEDURES_SCHEMA_UID = 'f7dc0a5b-16d0-4785-85e6-115e560ce975'

if any(v.startswith("<") for v in [PROJECT_UID,
    PREPARED_PATIENTS_UID, PREPARED_ENCOUNTERS_UID, PREPARED_PROCEDURES_UID,
    PREPARED_PATIENTS_SCHEMA_UID, PREPARED_ENCOUNTERS_SCHEMA_UID, PREPARED_PROCEDURES_SCHEMA_UID]):
    raise ValueError("Please fill in all UIDs from Tutorial 3 before continuing.")

print("Configuration loaded.")

Configuration loaded.


## Step 1 — Authenticate

In [2]:
my_username = "devex+site1admin@rhinofcp.com"  # REPLACE with your FCP username
session = rh.login(username=my_username, password=getpass(), rhino_api_url=ApiEnvironment.PROD_AWS_URL) # e.g., PROD_AWS_URL, STAGING_AWS_URL, DEV2_AWS_URL, SOLUTIONS_GCP_URL
print(f"Logged in as: {my_username}")

project   = session.project.get_projects(project_uids=[PROJECT_UID])[0]
workgroup = session.project.get_collaborating_workgroups(PROJECT_UID)[0]
WORKGROUP_UID = workgroup.uid

print(f"Project \t({project.uid}): \t{project.name}")
print(f"Workgroup \t({workgroup.uid}): \t{workgroup.name}")

Logged in as: devex+site1admin@rhinofcp.com
Project 	(a8d4dfc8-fcca-414d-a631-a4ae636263f6): 	Intro to Data Engineering - DevEx [DO NOT DELETE]
Workgroup 	(0f747b3f-f26a-4dbe-88ba-bfcbf6448cd4): 	DevEx Site 1


## Step 2 — Create OMOP Target Schemas

The three OMOP target schemas are defined as CSV files in the `schemas/` subfolder alongside this notebook. Each file describes the columns of one OMOP CDM table using the same transposed format used in Tutorial 1 (rows = attributes like `Variable Name`, `Type`; columns = fields).

The cell below registers them on the FCP via the SDK. If schemas with these names already exist in the project (e.g. from a previous run), they are reused automatically — nothing is overwritten.

**Why CSV files instead of inline code?**
Schema definitions are stable reference data, not runtime logic. Keeping them as CSV files makes them easy to inspect, diff in git, and reuse across projects without touching the notebook.

| File | Target Table | Source Dataset |
|---|---|---|
| `schemas/omop_person.csv` | `person` | Patients (prepared) |
| `schemas/omop_visit_occurrence.csv` | `visit_occurrence` | Encounters (prepared) |
| `schemas/omop_procedure_occurrence.csv` | `procedure_occurrence` | Procedures (prepared) |

In [3]:
from rhino_health.lib.endpoints.data_schema.data_schema_dataclass import DataSchemaCreateInput

# Extract workgroup UID
_wg_uid = WORKGROUP_UID.uid if hasattr(WORKGROUP_UID, "uid") else str(WORKGROUP_UID)
WORKGROUP_UID = _wg_uid

# add schemas dir at the same level as this notebook (assumes you're running the notebook from its current location)
_SCHEMA_DIR = os.path.join(os.getcwd(), "../schemas")

def _get_or_create_omop_schema(name, description, csv_filename):
    """Return UID of an existing schema by name, or create it from the CSV file."""
    existing = session.data_schema.get_data_schema_by_name(name, project_uid=PROJECT_UID)
    if existing:
        print(f"  '{name}' already exists — reusing {existing.uid}")
        return existing.uid
    schema = session.data_schema.create_data_schema(
        DataSchemaCreateInput(
            name=name,
            description=description,
            primary_workgroup_uid=_wg_uid,
            project_uid=PROJECT_UID,
            file_path=os.path.join(_SCHEMA_DIR, csv_filename),
        ),
        return_existing=False,
    )
    print(f"  '{name}' created — {schema.uid}")
    return schema.uid

print(f"Using workgroup UID: {_wg_uid}")
print("Creating OMOP target schemas...")
OMOP_PERSON_SCHEMA_UID = _get_or_create_omop_schema(
    name="OMOP Person",
    description="OMOP CDM Person table — target schema for patient demographic harmonization",
    csv_filename="omop_person.csv",
)
OMOP_VISIT_OCCURRENCE_SCHEMA_UID = _get_or_create_omop_schema(
    name="OMOP Visit Occurrence",
    description="OMOP CDM Visit Occurrence table — target schema for encounter harmonization",
    csv_filename="omop_visit_occurrence.csv",
)
OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID = _get_or_create_omop_schema(
    name="OMOP Procedure Occurrence",
    description="OMOP CDM Procedure Occurrence table — target schema for procedure harmonization",
    csv_filename="omop_procedure_occurrence.csv",
)

print(f"\nOMOP_PERSON_SCHEMA_UID               = '{OMOP_PERSON_SCHEMA_UID}'")
print(f"OMOP_VISIT_OCCURRENCE_SCHEMA_UID     = '{OMOP_VISIT_OCCURRENCE_SCHEMA_UID}'")
print(f"OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID = '{OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID}'")

Using workgroup UID: 0f747b3f-f26a-4dbe-88ba-bfcbf6448cd4
Creating OMOP target schemas...
  'OMOP Person' already exists — reusing 38b39062-4e90-4657-b19e-f6e95090d772
  'OMOP Visit Occurrence' already exists — reusing c7671311-cb15-4727-af15-b1285843600b
  'OMOP Procedure Occurrence' already exists — reusing 9d2a522b-7216-4958-a04f-4a582a9bbbb0

OMOP_PERSON_SCHEMA_UID               = '38b39062-4e90-4657-b19e-f6e95090d772'
OMOP_VISIT_OCCURRENCE_SCHEMA_UID     = 'c7671311-cb15-4727-af15-b1285843600b'
OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID = '9d2a522b-7216-4958-a04f-4a582a9bbbb0'


## Step 3 — Create Syntactic Mapping and Semantic Mapping Objects

- The **Syntactic Mapping** defines the structural transformation (which source columns map to which target fields)
- The **Semantic Mappings** define value-level vocabulary translations (e.g. `"Male"` → OMOP concept `8507`) and are created in **Needs Review** status, (pending approval)

Before creating the encounter mappings, we also create a **Custom Vocabulary** called `"Custom Encounter Type Codes"` to represent the source encounter type values used in the `TypeOfService` column. This custom vocabulary is then referenced in the semantic mapping for the encounters → OMOP Visit Occurrence table pair.

**Note on target_data_model_type:**
This tutorial uses `"omop"`. Change to `"fhir"` for FHIR targets or
`"custom"` for any non-standard target schema you have defined.

### 3a — Create Syntactic Mapping

A **Syntactic Mapping** defines how source data is structurally transformed into a target data model. Creating one involves three components:

1. **`global_configuration`** — sets up the mapping context:
   - `data_sources_to_data_schemas` — maps a snake_case source name (e.g. `"patients"`) to its prepared source schema UID. This tells the mapping engine which schema describes each input dataset.
   - `vocabulary_mapping` — lists any vocabularies used for semantic lookups (e.g. OMOP concept IDs). Left empty here and populated in Step 2b when semantic mappings are created.

2. **`target_data_model_type`** — the target standard: `OMOP`, `FHIR`, or `CUSTOM`.

3. **`table_configurations`** — one entry per target OMOP table, each with:
   - `table_name` — the OMOP target table (e.g. `"person"`, `"visit_occurrence"`, `"procedure_occurrence"`)
   - `field_configurations` — the column-level transformation rules. Left empty here; auto-generated in a later step.

The result is a single syntactic mapping object called `"Data to OMOP"` that covers all three source datasets in one run.

In [4]:
from rhino_health.lib.endpoints.syntactic_mapping.syntactic_mapping_dataclass import (
    SyntacticMappingCreateInput, SyntacticMappingDataModel,
    SyntacticMappingConfig, GlobalConfiguration, TableConfiguration
)
import json as _json

_SM_NAME = "Map All Prepared Data to OMOP"

# Delete ALL existing syntactic mappings with this name (including locked versions
# from prior runs). A mapping becomes locked once used in a code run, so we must
# start fresh each time this cell executes.
_existing_sms = session.syntactic_mapping.search_for_syntactic_mappings_by_name(
    _SM_NAME, project_uid=PROJECT_UID
)
for _sm in _existing_sms:
    try:
        session.syntactic_mapping.remove_syntactic_mapping(_sm.uid)
        print(f"Deleted existing syntactic mapping: {_sm.uid}")
    except Exception as _del_err:
        print(f"  Warning: could not delete {_sm.uid}: {_del_err}")

# output_table_names_to_data_schemas tells the platform how many output directories
# to create on the client (/output/0/, /output/1/, /output/2/) and which OMOP schema
# each output table conforms to. Without this, the container crashes with:
#   OSError: Cannot save file into a non-existent directory: '/output/0'
_sm_input = SyntacticMappingCreateInput(
    name=_SM_NAME,
    primary_workgroup_uid=WORKGROUP_UID,
    project_uid=PROJECT_UID,
    target_data_model_type=SyntacticMappingDataModel.OMOP,
    mapping_config=SyntacticMappingConfig(
        global_configuration=GlobalConfiguration(
            data_sources_to_data_schemas=[
                {"patients":   PREPARED_PATIENTS_SCHEMA_UID},
                {"encounters": PREPARED_ENCOUNTERS_SCHEMA_UID},
                {"procedures": PREPARED_PROCEDURES_SCHEMA_UID},
            ],
            vocabulary_mapping=[],
            output_table_names_to_data_schemas=[
                {"person":               OMOP_PERSON_SCHEMA_UID},
                {"visit_occurrence":     OMOP_VISIT_OCCURRENCE_SCHEMA_UID},
                {"procedure_occurrence": OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID},
            ],
        ),
        table_configurations=[
            TableConfiguration(table_name="person",               field_configurations=[]),
            TableConfiguration(table_name="visit_occurrence",     field_configurations=[]),
            TableConfiguration(table_name="procedure_occurrence",  field_configurations=[]),
        ],
    ),
)

# The SDK has no field for target_data_model_version, but the platform requires
# it to validate OMOP table names against — omitting it causes every table name
# to fail validation with "Table X is not part of omop None" (the trailing
# "None" is the unset version). Create via the raw API so we can include it.
# model_dump_json serializes Enum -> string value; json.loads gives a plain dict
# (plain .dict()/model_dump() would leave target_data_model_type as an Enum,
# which doesn't serialize the way the backend expects).
_payload = _json.loads(_sm_input.model_dump_json(by_alias=True))
_payload["target_data_model_version"] = "5.4"

_resp = session.post("/syntactic_mappings", data=_payload).raw_response.json()
SYNTACTIC_MAPPING_UID = _resp["data"]["uid"]
print(f"Syntactic mapping created: {_SM_NAME}")
print(f"  UID: {SYNTACTIC_MAPPING_UID}")
print(f"  OMOP version: {_resp['data'].get('target_data_model_version')}")
print(f"  Note: field configurations are populated in Step 5.")

Deleted existing syntactic mapping: b56beaa8-f5d9-49ca-adb5-45a906398618
Syntactic mapping created: Map All Prepared Data to OMOP
  UID: 872bcd99-92cb-4bc1-bb52-a30065fa6561
  OMOP version: 5.4
  Note: field configurations are populated in Step 5.


### 3b — Create Semantic Mappings

A **Semantic Mapping** translates individual source values to target vocabulary concepts (e.g., `"Asian"` → OMOP concept `8515`). Each mapping links an **input vocabulary** (custom — the unique source values for one column) to an **output vocabulary** (the OMOP standard vocabulary on the platform).

This cell does three things in sequence:
1. **Discovers** the OMOP standard vocabulary already on the platform
2. **Creates custom input vocabularies** — one per source column, populated with the known source values
3. **Creates semantic mappings** — pairs each input vocabulary with the OMOP output vocabulary, then waits for the AI term-matching to finish

Five semantic mappings are created (Gender, Race, Ethnicity, Visit Type, CPT4 codes) covering all three source datasets.

In [5]:
from rhino_health.lib.endpoints.semantic_mapping.semantic_mapping_dataclass import (
    SemanticMappingCreateInput, DatasetColumn, VocabularyInput,
)

# ── 1. Discover domain-specific OMOP standard vocabularies via raw API ────────
# IMPORTANT: The platform does NOT support scoping AI term-matching by passing
# the generic "OMOP" vocabulary (which covers all ~49 domains) together with
# output_vocabulary_categories as a filter. That combination searches the full
# unscoped concept space and reliably fails to confidently match most entries
# (observed as "N entries failed to process" / processing_status "Error" on
# every mapping, regardless of how correctly the category string is spelled).
#
# Instead, the platform exposes each OMOP domain as its OWN separate standard
# vocabulary (e.g. a vocabulary literally named "Procedure", another named
# "Gender", etc.) — confirmed by comparing a known-working semantic mapping
# (input_vocabulary == output_vocabulary == the standard "Procedure" vocabulary,
# data_model="omop") against a failing one (both pointing at the generic "OMOP"
# vocabulary, data_model=""). Use these domain-specific vocabularies as the
# output_vocabulary_uid for each mapping, with no categories filter needed.
print("Discovering domain-specific OMOP standard vocabularies...")

def _discover_standard_vocab(name):
    raw = session.get("/vocabularies", params={"name": name, "type": "standard"})
    matches = [v for v in raw.raw_response.json().get("data", []) if v.get("name") == name]
    if not matches:
        raise RuntimeError(f"No standard vocabulary named '{name}' found.")
    if len(matches) > 1:
        print(f"  Warning: multiple standard vocabularies named '{name}', using the first: {matches}")
    uid = matches[0]["uid"]
    print(f"  '{name}' -> {uid}")
    return uid

GENDER_DOMAIN_VOCAB_UID     = _discover_standard_vocab("Gender")
RACE_DOMAIN_VOCAB_UID       = _discover_standard_vocab("Race")
ETHNICITY_DOMAIN_VOCAB_UID  = _discover_standard_vocab("Ethnicity")
VISIT_DOMAIN_VOCAB_UID      = _discover_standard_vocab("Visit")
PROCEDURE_DOMAIN_VOCAB_UID  = _discover_standard_vocab("Procedure")

# ── 2. Helpers ────────────────────────────────────────────────────────────────
def _create_source_vocab(name, terms):
    """Create a fresh custom vocabulary of source terms. Returns UID.

    Like semantic/syntactic mappings, a vocabulary can become locked once it's
    referenced by a semantic mapping used in a code run. return_existing=True
    would silently hand back a locked/stale vocabulary on a notebook re-run, so
    we delete any existing vocabulary with this name and create fresh every time.
    """
    _existing_vocabs = session.vocabulary.search_for_vocabularies_by_name(
        name, project_uid=PROJECT_UID
    )
    for _v in _existing_vocabs:
        try:
            session.vocabulary.remove_vocabulary(_v.uid)
            print(f"  Deleted existing vocabulary '{name}': {_v.uid}")
        except Exception as _del_err:
            print(f"  Warning: could not delete {_v.uid}: {_del_err}")

    vocab = session.vocabulary.create_vocabulary(
        VocabularyInput(
            name=name,
            primary_workgroup_uid=WORKGROUP_UID,
            project_uid=PROJECT_UID,
            terms=[{"term_identifier": t, "term_display_name": t} for t in terms],
        ),
        return_existing=False,
    )
    print(f"  vocab '{name}': {vocab.uid}")
    return vocab.uid

def _create_semantic_mapping(name, input_vocab_uid, output_vocab_uid, dataset_uid=None, field_name=None):
    """Create a fresh semantic mapping. Returns the SemanticMapping object.

    output_vocab_uid: the domain-specific OMOP standard vocabulary (e.g. the
    "Procedure" vocabulary), NOT the generic "OMOP" vocabulary — see note above.

    dataset_uid / field_name: when provided, the platform scans the actual dataset
    column to seed source term suggestions alongside the vocabulary terms.
    NOTE: passing source_dataset_columns=[] or None causes a backend 500 ("list index
    out of range") — always provide a real dataset column or omit both arguments.

    Like the syntactic mapping in Step 3a, a semantic mapping becomes locked
    ("not editable as it is used in a code run") once it's been consumed by a
    harmonization run. return_existing=True would silently hand back that locked
    object on a notebook re-run, so we delete any existing mapping with this name
    and create a fresh one every time this cell executes.
    """
    _existing_sms = session.semantic_mapping.search_for_semantic_mappings_by_name(
        name, project_uid=PROJECT_UID
    )
    for _sm in _existing_sms:
        try:
            session.semantic_mapping.remove_semantic_mapping(_sm.uid)
            print(f"  Deleted existing semantic mapping '{name}': {_sm.uid}")
        except Exception as _del_err:
            print(f"  Warning: could not delete {_sm.uid}: {_del_err}")

    source_cols = (
        [DatasetColumn(dataset_uid=dataset_uid, field_name=field_name)]
        if dataset_uid and field_name
        else []
    )
    sm = session.semantic_mapping.create_semantic_mapping(
        SemanticMappingCreateInput(
            name=name,
            primary_workgroup_uid=WORKGROUP_UID,
            project_uid=PROJECT_UID,
            input_vocabulary_uid=input_vocab_uid,
            output_vocabulary_uid=output_vocab_uid,
            source_dataset_columns=source_cols,
        ),
        return_existing=False,
    )
    print(f"  mapping '{name}': {sm.uid}  [{sm.processing_status.value}]")
    return sm

# ── 3. Create source vocabularies ────────────────────────────────────────────
print("\nCreating source vocabularies...")
_gender_vocab_uid    = _create_source_vocab("Source: Gender Values",     ["Male", "Female", "Other"])
_race_vocab_uid      = _create_source_vocab("Source: Race Values",       ["Asian", "Black", "White", "Other"])
_ethnicity_vocab_uid = _create_source_vocab("Source: Ethnicity Values",  ["Hispanic", "Non-Hispanic"])
_visit_vocab_uid     = _create_source_vocab("Source: Visit Type Values", ["Outpatient", "Inpatient", "Emergency"])
_cpt_vocab_uid       = _create_source_vocab("Source: CPT4 Codes",        ["45378", "44950", "99203", "99212", "99213", "99285"])

# ── 4. Create semantic mappings ───────────────────────────────────────────────
# NOTE on CPT: The prepared procedures dataset stores ProcedureCode as floats
# (e.g. 99285.0) because pandas reads numeric columns that way. When the platform
# scans the column via source_dataset_columns, it creates entries keyed by float
# strings ("99285.0"). The source vocabulary terms produce entries keyed by int
# strings ("99285"). Both entry types exist after processing.
#
# At harmonization time, the syntactic mapping applies _normalize_cpt
# (99285.0 → "99285") BEFORE the semantic lookup, so the lookup key is int format.
# Step 4 therefore approves BOTH formats so all entries are approved and the
# overall mapping status reaches "Approved".
#
# source_dataset_columns must point to a real column — passing [] or None causes
# a backend 500 ("list index out of range") regardless of SDK version.
print("\nCreating semantic mappings...")
_gender_sm    = _create_semantic_mapping("Normalize Gender [OMOP]",     _gender_vocab_uid,    GENDER_DOMAIN_VOCAB_UID,     PREPARED_PATIENTS_UID,    "Gender")
_race_sm      = _create_semantic_mapping("Normalize Race [OMOP]",       _race_vocab_uid,      RACE_DOMAIN_VOCAB_UID,       PREPARED_PATIENTS_UID,    "Race")
_ethnicity_sm = _create_semantic_mapping("Normalize Ethnicity [OMOP]",  _ethnicity_vocab_uid, ETHNICITY_DOMAIN_VOCAB_UID,  PREPARED_PATIENTS_UID,    "Ethnicity")
_visit_sm     = _create_semantic_mapping("Normalize Visit Type [OMOP]", _visit_vocab_uid,     VISIT_DOMAIN_VOCAB_UID,      PREPARED_ENCOUNTERS_UID,  "TypeOfService")
_cpt_sm       = _create_semantic_mapping("Normalize CPT4 Codes [OMOP]", _cpt_vocab_uid,       PROCEDURE_DOMAIN_VOCAB_UID,  PREPARED_PROCEDURES_UID,  "ProcedureCode")

GENDER_SM_UID     = _gender_sm.uid
RACE_SM_UID       = _race_sm.uid
ETHNICITY_SM_UID  = _ethnicity_sm.uid
VISIT_TYPE_SM_UID = _visit_sm.uid
CPT_SM_UID        = _cpt_sm.uid

# Also export vocabulary UIDs — needed by Step 4 to populate GlobalConfiguration.vocabulary_mapping
GENDER_VOCAB_UID     = _gender_vocab_uid
RACE_VOCAB_UID       = _race_vocab_uid
ETHNICITY_VOCAB_UID  = _ethnicity_vocab_uid
VISIT_TYPE_VOCAB_UID = _visit_vocab_uid
CPT_VOCAB_UID        = _cpt_vocab_uid

# ── 5. Wait for AI term-matching to complete ──────────────────────────────────
print("\nWaiting for AI term-matching (this may take several minutes per mapping)...")
for uid, label in [
    (GENDER_SM_UID,     "Gender"),
    (RACE_SM_UID,       "Race"),
    (ETHNICITY_SM_UID,  "Ethnicity"),
    (VISIT_TYPE_SM_UID, "Visit Type"),
    (CPT_SM_UID,        "CPT4"),
]:
    current = session.semantic_mapping.get_semantic_mapping(uid)
    if not current._finished_processing:
        current = current.wait_for_completion(timeout_seconds=900, print_progress=False)
    suffix = " (already done)" if current.processing_status.value != "Error" else ""
    print(f"  {label}: {current.processing_status.value}{suffix}")
    if current.processing_status.value == "Error":
        print(f"    error detail: {current.processing_error_message}")

print("\nAll semantic mappings ready for review — continue to Step 4.")

Discovering domain-specific OMOP standard vocabularies...
  'Gender' -> 1a7816e4-f60f-402d-bb59-12bc1bbb31ac
  'Race' -> 3ff1864c-367d-4e5a-a002-684e29ebd32b
  'Ethnicity' -> 09404741-a866-47c4-9cc3-607644072bf2
  'Visit' -> cde30fe2-29b8-40f0-ba10-f0c784f8bf62
  'Procedure' -> 7e9880b0-49b9-4182-bcd7-516b280f0228

Creating source vocabularies...
  Deleted existing vocabulary 'Source: Gender Values': cdd61c17-e5f7-4a1e-8b61-cf18c0033c5f
  vocab 'Source: Gender Values': 618bcf8b-002c-4d37-aac6-6013c520d9b6
  Deleted existing vocabulary 'Source: Race Values': 755e4f78-03ed-4de2-9f0e-c2191d9c64cf
  vocab 'Source: Race Values': 17a86939-ffda-448f-9c75-856648f537c6
  Deleted existing vocabulary 'Source: Ethnicity Values': 100b424b-5ca8-4dcb-8077-b8172cd91152
  vocab 'Source: Ethnicity Values': 546dfb03-557f-4ed4-8df0-1e02d47ca25f
  Deleted existing vocabulary 'Source: Visit Type Values': e8d5d7a8-712e-4597-bf86-b0e89833e1b1
  vocab 'Source: Visit Type Values': aff8c42b-8bec-49fd-bfc9-74dbbc

## Step 4 — Review and Approve Semantic Mappings

The platform has proposed AI-generated term matches for each vocabulary.
**This step requires human review.** Incorrect term matches will silently
produce wrong concept IDs in the harmonized output.

## **Approve these manually in the Dashboard.** 
`approve_mappings()` does accept
a freshly client-generated `entry_uid`, but doing so creates an orphaned entry
not linked to the mapping's own tracked unique-value set -- it has no effect
on `approved_values_count`/`unique_approved_entries_count` and the mapping
never reaches "Approved" this way (confirmed empirically). There is no public
SDK endpoint to fetch the real entry UIDs either, so approval has to happen
where it's actually supported:

Dashboard -> Projects -> Data Mappings -> click each semantic mapping below ->
review/approve the proposed term for **every** source value shown (including
both float- and int-format CPT codes if both appear, e.g. "99285.0" and
"99285" -- `_normalize_cpt` in Step 5 converts to int format before the
lookup, but the mapping's overall status requires every detected unique value
to be resolved), using these reference concept IDs (from the Tutorial 4 README):

- Gender: Male→8507, Female→8532, Other→8521
- Race: Asian→8515, Black→8516, White→8527, Other→8522
- Ethnicity: Hispanic→38003563, Non-Hispanic→38003564
- Visit: Outpatient→9202, Inpatient→9201, Emergency→9203
- CPT: 45378→4287782, 44950→4196867, 99203→4098498, 99212→4098460, 99213→4098462, 99285→4129922

Once every mapping shows **Approved** with 100% coverage in the Dashboard,
run the cell below to confirm before moving on to Step 5. (Step 5 also sets a
safety net -- unresolved/unapproved values fall back to OMOP concept_id `0`
instead of crashing the harmonization run.)

In [6]:
print("Checking semantic mapping approval status...\n")

_SEMANTIC_MAPPINGS_TO_APPROVE = [
    (GENDER_SM_UID,     "Gender"),
    (RACE_SM_UID,       "Race"),
    (ETHNICITY_SM_UID,  "Ethnicity"),
    (VISIT_TYPE_SM_UID, "Visit Type"),
    (CPT_SM_UID,        "CPT4"),
]

_all_approved = True
for uid, label in _SEMANTIC_MAPPINGS_TO_APPROVE:
    current = session.semantic_mapping.get_semantic_mapping(uid)
    status = current.processing_status.value
    marker = "OK" if status == "Approved" else "PENDING"
    coverage = current.semantic_mapping_info or {}
    print(f"  [{marker}] {label}: {status}  "
          f"({coverage.get('unique_approved_entries_count', '?')}/{coverage.get('num_unique_values', '?')} unique terms approved, "
          f"covering {coverage.get('approved_values_count', '?')}/{coverage.get('value_count', '?')} total values)")
    if status != "Approved":
        _all_approved = False

if _all_approved:
    print("\nAll semantic mappings approved. Continue to Step 5.")
else:
    print("\nNot all mappings are approved yet -- approve the pending ones in the "
          "Dashboard (Projects -> Data Mappings), then rerun this cell.")

Checking semantic mapping approval status...

  [OK] Gender: Approved  (2/2 unique terms approved, covering 95/95 total values)
  [OK] Race: Approved  (4/4 unique terms approved, covering 95/95 total values)
  [OK] Ethnicity: Approved  (2/2 unique terms approved, covering 95/95 total values)
  [OK] Visit Type: Approved  (3/3 unique terms approved, covering 98/98 total values)
  [OK] CPT4: Approved  (6/6 unique terms approved, covering 98/98 total values)

All semantic mappings approved. Continue to Step 5.


## Step 5 — Auto-generate Syntactic Mappings

With all semantic mappings approved, auto-populate the field-level transformation
rules for each syntactic mapping. This creates rules for every source→target
column pair based on the approved semantic mappings and schema alignment.
You can review and manually adjust these rules in the FCP Dashboard if needed.

In [7]:
from rhino_health.lib.endpoints.syntactic_mapping.syntactic_mapping_dataclass import (
    SyntacticMappingConfig, GlobalConfiguration, TableConfiguration,
    FieldConfiguration, SourceDataField, VocabularyMapping,
    SourceValueTransformation, SecureUUIDTransformation,
    SpecificValueTransformation, SemanticMappingTransformation, SemanticMappingTransformationOutput,
    SemanticMappingTransformationOnMissing, SemanticMappingTransformationOnUnapprovedValue,
    RowPythonCodeTransformation, RowPythonCodeTransformationOnError,
)
from rhino_health.lib.endpoints.semantic_mapping.semantic_mapping_dataclass import VocabularyType
import json as _json

print("Updating syntactic mapping with OMOP field configurations...")

# RowPythonCodeTransformation exec namespace:
#   - input variable: `row`  (the scalar value from the previous step)
#   - output variable: `output`  (what the executor reads back — NOT `result`)
#   - also available: np, pd, arrow

# Integer ID: hash source value to a consistent 31-bit integer for OMOP ID columns.
_INT_ID_CODE = (
    "import hashlib; "
    "output = abs(int(hashlib.sha256(str(row).encode()).hexdigest(), 16)) % (2**31)"
)
_int_id = lambda: [
    SourceValueTransformation(),
    RowPythonCodeTransformation(code=_INT_ID_CODE, on_error=RowPythonCodeTransformationOnError.FAIL),
]

# NOTE: Gender/TypeOfService raw values were confirmed (via num_unique_values
# exactly matching the clean category count, and the Dashboard's own display of
# the raw detected value) to have NO case variation in practice -- a prior
# "_normalize" (str.capitalize()) step here was removed because it transformed
# the lookup value AWAY from what the Dashboard-approved entry is keyed to
# (the literal raw value), causing the semantic mapping lookup to silently
# fall through and write the matched target_term_name text instead of the
# numeric target_identifier.

# CPT code normalization: pandas reads numeric codes as floats (99285.0).
# Convert to clean integer string ("99285") so the vocabulary lookup matches.
_normalize_cpt = RowPythonCodeTransformation(
    code="output = str(int(float(row)))",
    on_error=RowPythonCodeTransformationOnError.FAIL,
)

updated_config = SyntacticMappingConfig(
    global_configuration=GlobalConfiguration(
        data_sources_to_data_schemas=[
            {"patients":   PREPARED_PATIENTS_SCHEMA_UID},
            {"encounters": PREPARED_ENCOUNTERS_SCHEMA_UID},
            {"procedures": PREPARED_PROCEDURES_SCHEMA_UID},
        ],
        # Each vocabulary is the domain-specific OMOP standard vocabulary
        # discovered in Step 3b (e.g. the "Procedure" vocabulary for cpt) —
        # the same vocabulary used as output_vocabulary_uid for the matching
        # semantic mapping, not the generic "OMOP" vocabulary.
        vocabulary_mapping=[
            VocabularyMapping(name="gender_vocab",     type=VocabularyType.STANDARD, uid=GENDER_DOMAIN_VOCAB_UID,    metadata={"semantic_mapping_uid": GENDER_SM_UID}),
            VocabularyMapping(name="race",        type=VocabularyType.STANDARD, uid=RACE_DOMAIN_VOCAB_UID,      metadata={"semantic_mapping_uid": RACE_SM_UID}),
            VocabularyMapping(name="ethnicity",   type=VocabularyType.STANDARD, uid=ETHNICITY_DOMAIN_VOCAB_UID, metadata={"semantic_mapping_uid": ETHNICITY_SM_UID}),
            VocabularyMapping(name="visit_type_vocab",  type=VocabularyType.STANDARD, uid=VISIT_DOMAIN_VOCAB_UID,     metadata={"semantic_mapping_uid": VISIT_TYPE_SM_UID}),
            VocabularyMapping(name="cpt",         type=VocabularyType.STANDARD, uid=PROCEDURE_DOMAIN_VOCAB_UID, metadata={"semantic_mapping_uid": CPT_SM_UID}),
        ],
        output_table_names_to_data_schemas=[
            {"person":               OMOP_PERSON_SCHEMA_UID},
            {"visit_occurrence":     OMOP_VISIT_OCCURRENCE_SCHEMA_UID},
            {"procedure_occurrence": OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID},
        ],
    ),
    table_configurations=[
        # ── person (source: patients) ─────────────────────────────────────
        TableConfiguration(
            table_name="person",
            field_configurations=[
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="patientID")],
                    target_field="person_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="Gender")],
                    target_field="gender_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SemanticMappingTransformation(
                            vocabulary_name="gender_vocab",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                            on_unapproved_value=SemanticMappingTransformationOnUnapprovedValue.TREAT_AS_MISSING,
                            on_missing=SemanticMappingTransformationOnMissing.SET_TO_DEFAULT,
                            default_value=0,  # OMOP sentinel for "no matching concept"
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="YearOfBirth")],
                    target_field="year_of_birth",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="Race")],
                    target_field="race_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SemanticMappingTransformation(
                            vocabulary_name="race",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                            on_unapproved_value=SemanticMappingTransformationOnUnapprovedValue.TREAT_AS_MISSING,
                            on_missing=SemanticMappingTransformationOnMissing.SET_TO_DEFAULT,
                            default_value=0,  # OMOP sentinel for "no matching concept"
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="Ethnicity")],
                    target_field="ethnicity_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SemanticMappingTransformation(
                            vocabulary_name="ethnicity",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                            on_unapproved_value=SemanticMappingTransformationOnUnapprovedValue.TREAT_AS_MISSING,
                            on_missing=SemanticMappingTransformationOnMissing.SET_TO_DEFAULT,
                            default_value=0,  # OMOP sentinel for "no matching concept"
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="patientID")],
                    target_field="person_source_value",
                    transformations=[SourceValueTransformation()],
                ),
            ],
        ),
        # ── visit_occurrence (source: encounters) ─────────────────────────
        TableConfiguration(
            table_name="visit_occurrence",
            field_configurations=[
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="visitID")],
                    target_field="visit_occurrence_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="patientID")],
                    target_field="person_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="TypeOfService")],
                    target_field="visit_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SemanticMappingTransformation(
                            vocabulary_name="visit_type_vocab",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                            on_unapproved_value=SemanticMappingTransformationOnUnapprovedValue.TREAT_AS_MISSING,
                            on_missing=SemanticMappingTransformationOnMissing.SET_TO_DEFAULT,
                            default_value=0,  # OMOP sentinel for "no matching concept"
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="DateOfService")],
                    target_field="visit_start_date",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="DateOfService")],
                    target_field="visit_end_date",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="visitID")],
                    target_field="visit_type_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SpecificValueTransformation(value=32817),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="visitID")],
                    target_field="visit_source_value",
                    transformations=[SourceValueTransformation()],
                ),
            ],
        ),
        # ── procedure_occurrence (source: procedures) ─────────────────────
        TableConfiguration(
            table_name="procedure_occurrence",
            field_configurations=[
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="visitID")],
                    target_field="procedure_occurrence_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="patientID")],
                    target_field="person_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="ProcedureCode")],
                    target_field="procedure_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        _normalize_cpt,  # 99285.0 → "99285"
                        SemanticMappingTransformation(
                            vocabulary_name="cpt",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                            on_unapproved_value=SemanticMappingTransformationOnUnapprovedValue.TREAT_AS_MISSING,
                            on_missing=SemanticMappingTransformationOnMissing.SET_TO_DEFAULT,
                            default_value=0,  # OMOP sentinel for "no matching concept"
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="ProcedureDate")],
                    target_field="procedure_date",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="visitID")],
                    target_field="procedure_type_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SpecificValueTransformation(value=32817),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="ProcedureCode")],
                    target_field="procedure_source_value",
                    transformations=[
                        SourceValueTransformation(),
                        _normalize_cpt,  # 99285.0 → "99285"
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="visitID")],
                    target_field="visit_occurrence_id",
                    transformations=_int_id(),
                ),
            ],
        ),
    ],
)

# Bypass partial_update's exclude_unset=True (strips transformation_type discriminators).
# model_dump_json serializes Enum → string value; json.loads gives a plain dict.
config_payload = _json.loads(updated_config.model_dump_json(by_alias=True))
session.patch(
    f"/syntactic_mappings/{SYNTACTIC_MAPPING_UID}",
    data={"mapping_config": config_payload},
)
print("  Done — syntactic mapping updated with field configurations.")

Updating syntactic mapping with OMOP field configurations...
  Done — syntactic mapping updated with field configurations.


## Step 6 — Run Harmonization

In [8]:
from rhino_health.lib.endpoints.syntactic_mapping.syntactic_mapping_dataclass import DataHarmonizationRunInput

print("Starting OMOP harmonization run (all three tables)...")
async_response = session.syntactic_mapping.run_data_harmonization(
    SYNTACTIC_MAPPING_UID,
    DataHarmonizationRunInput(
        input_dataset_uids=[PREPARED_PATIENTS_UID, PREPARED_ENCOUNTERS_UID, PREPARED_PROCEDURES_UID],
        semantic_mapping_uids_by_vocabularies={
            "gender_vocab":     GENDER_SM_UID,
            "race":             RACE_SM_UID,
            "ethnicity":        ETHNICITY_SM_UID,
            "visit_type_vocab": VISIT_TYPE_SM_UID,
            "cpt":              CPT_SM_UID,
        },
        timeout_seconds=900,
    )
)
code_run_uid = async_response.code_run_uid
print(f"  Run started (code_run_uid={code_run_uid})")

code_run = session.code_run.get_code_run(code_run_uid)
harmonization_result = code_run.wait_for_completion(timeout_seconds=900, print_progress=True)
print(f"\nHarmonization run status: {harmonization_result.status.value}")

if harmonization_result.status.value != "Completed":
    print("Errors:", harmonization_result.errors)
    print("Warnings:", harmonization_result.warnings)
    raise RuntimeError(
        f"Harmonization run did not complete successfully "
        f"(status={harmonization_result.status.value}). See errors/warnings above."
    )

if harmonization_result.warnings:
    print("Warnings during run:", harmonization_result.warnings)

# Match output datasets to OMOP tables by data_schema_uid rather than assuming a
# fixed nesting/order in output_dataset_uids -- more robust than indexing
# .output_dataset_uids.root[...] directly, which breaks if a run produces a
# different number/order of outputs than expected.
_output_datasets = harmonization_result.output_datasets or []
_schema_to_var = {
    OMOP_PERSON_SCHEMA_UID:               "OMOP_PERSON_UID",
    OMOP_VISIT_OCCURRENCE_SCHEMA_UID:     "OMOP_VISIT_UID",
    OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID: "OMOP_PROCEDURE_UID",
}
_resolved = {}
_unmatched = []
for ds in _output_datasets:
    var_name = _schema_to_var.get(ds.data_schema_uid)
    if var_name:
        _resolved[var_name] = ds.uid
    else:
        _unmatched.append(ds)

_missing = [name for name in ("OMOP_PERSON_UID", "OMOP_VISIT_UID", "OMOP_PROCEDURE_UID") if name not in _resolved]
if _missing or _unmatched:
    print(f"Produced {len(_output_datasets)} output dataset(s): {[(d.name, d.uid) for d in _output_datasets]}")
    raise RuntimeError(
        f"Could not match all expected OMOP tables to output datasets. "
        f"Missing: {_missing}. Unmatched output datasets: {[(d.name, d.uid) for d in _unmatched]}"
    )

OMOP_PERSON_UID    = _resolved["OMOP_PERSON_UID"]
OMOP_VISIT_UID     = _resolved["OMOP_VISIT_UID"]
OMOP_PROCEDURE_UID = _resolved["OMOP_PROCEDURE_UID"]

print(f"\n  OMOP Person UID:               {OMOP_PERSON_UID}")
print(f"  OMOP Visit Occurrence UID:     {OMOP_VISIT_UID}")
print(f"  OMOP Procedure Occurrence UID: {OMOP_PROCEDURE_UID}")

Starting OMOP harmonization run (all three tables)...
  Run started (code_run_uid=29a85a1b-fafe-4e3a-880d-c12c936298d2)
Waiting for code run to complete (0 hours 0 minutes and 0 seconds)
Done.

Harmonization run status: Failed
Errors: ['Failed to import dataset: ValueError: Unable to parse string "FEMALE" at position 0', 'Failed to import dataset: ValueError: Unable to parse string "Outpatient Visit" at position 0']
Warnings: []


RuntimeError: Harmonization run did not complete successfully (status=Failed). See errors/warnings above.

## Step 7 — Verify Output Datasets

In [ ]:
from rhino_health.lib.metrics import Count, Mean

# Row counts — Count requires a variable (any column will do)
table_id_cols = {
    OMOP_PERSON_UID:    "person_id",
    OMOP_VISIT_UID:     "visit_occurrence_id",
    OMOP_PROCEDURE_UID: "procedure_occurrence_id",
}
outputs = [
    (OMOP_PERSON_UID,    "OMOP Person"),
    (OMOP_VISIT_UID,     "OMOP Visit Occurrence"),
    (OMOP_PROCEDURE_UID, "OMOP Procedure Occurrence"),
]

print(f"{'Table':<35} {'Rows':>8}  UID")
print("─" * 80)
for ds_uid, label in outputs:
    try:
        result = session.dataset.get_dataset_metric(ds_uid, Count(variable=table_id_cols[ds_uid]))
        count = result.output.get("count", result.output)
        print(f"{label:<35} {count:>8,}  {ds_uid}")
    except Exception as e:
        print(f"{label:<35} ERROR: {e}")

# Spot-check: mean of concept ID columns — non-zero confirms semantic mapping produced OMOP IDs
print("\nSpot check — mean of key OMOP concept ID columns:")
checks = [
    (OMOP_PERSON_UID,    "gender_concept_id",   "Person"),
    (OMOP_PERSON_UID,    "race_concept_id",      "Person"),
    (OMOP_PERSON_UID,    "ethnicity_concept_id", "Person"),
    (OMOP_VISIT_UID,     "visit_concept_id",     "Visit"),
    (OMOP_PROCEDURE_UID, "procedure_concept_id", "Procedure"),
]
print(f"  {'Table':<12} {'Column':<25} {'Mean concept_id':>18}")
print("  " + "─" * 58)
for ds_uid, col, tbl in checks:
    try:
        result = session.dataset.get_dataset_metric(ds_uid, Mean(variable=col))
        mean_val = result.output.get("mean", result.output)
        flag = "  ← check mapping" if mean_val is None or mean_val == 0 else ""
        print(f"  {tbl:<12} {col:<25} {mean_val:>18.2f}{flag}")
    except Exception as e:
        print(f"  {tbl:<12} {col:<25} ERROR: {e}")

## FCP UI — What to Check After Running This Notebook

1. **Harmonization Mappings** → Dashboard → Select Your Project → Data Mappings
   - Under "Syntactic Mappings" - you should see 1 Syntactic Mapping with a Target Data Model of OMOP, 3 source schemas, and 3 target tables
      - Click on the mapping: all required target fields should be linked to a source field
   - Under "Semantic Mappings" - you should see 5 Semantic Mappings
      - Each should have a Status of "Approved", with 100% Mapping Coverage
      - Click on any mapping to see a list of SOURCE VALUE(s) to SELECTED TARGET VALUE(s), with approval state and confidence level
   - Under "Custom Vocabularies" - you should see 5 Custom Vocabularies
      - Click on any mapping to see a list of CODE(s) and corresponding DISPLAY NAME(s)

2. **Harmonization Runs** → Dashboard → Select Your Project → Code Runs
   - At least 1 harmonization run (Type DH) appears under the relevant Code Object
   - Click on run → Logs tab → check for warnings about unmapped values

3. **Output Datasets** → Dashboard → Select Your Project → Datasets
   - Three new OMOP datasets appear (person, visit_occurrence, procedure_occurrence)
   - Click any → Analytics tab → inspect distributions
   - A non-zero concept_id=0 rate means some source values were not covered
     by your semantic mapping — return to Step 4 to add missing terms

## Summary — Copy These UIDs

In [ ]:
print("=" * 60)
print("  Tutorial 4 Complete — save these UIDs")
print("=" * 60)
print(f"OMOP_PERSON_UID    = '{OMOP_PERSON_UID}'")
print(f"OMOP_VISIT_UID     = '{OMOP_VISIT_UID}'")
print(f"OMOP_PROCEDURE_UID = '{OMOP_PROCEDURE_UID}'")
print("=" * 60)
print("\nContinue to: Tutorial 5 - Validation")